In [3]:
import pandas as pd
import numpy as np
import commodity_common_functions as CCF
import seaborn as sns
import dataframe_image as dfi
from pathlib import Path
import json
import seaborn as sns
import datetime as dt
import logging
from datetime import datetime
import os
import OptionRiskMetrixETF as ORME
import OptionRiskMetrix as ORM
import RiskManagementSystem as RMS
import matplotlib.pyplot as plt

In [4]:
def findUnderlyingOptionInfo(date):
    SQL_cmd = f"""
    SELECT underlying_instr_id,instrument_id
    FROM daily_data_option
    WHERE trading_date = '{date}'
    ORDER BY open_interest DESC
    """
    max_open_interest = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    max_open_interest = (
        max_open_interest.groupby("underlying_instr_id").first().reset_index()
    )
    max_open_interest["underlying_instr_id"] = max_open_interest[
        "underlying_instr_id"
    ].apply(
        lambda x: (
            "IH" + x[2:]
            if x.startswith("HO")
            else (
                "IF" + x[2:]
                if x.startswith("IO")
                else ("IM" + x[2:] if x.startswith("MO") else x)
            )
        )
    )
    SQL_cmd = f"""
    SELECT         
        i.underlying_instr_id,
        i.exchange_id,
        i.expiredate,
        i.volume_multiple,
        i.price_tick,
        icr.open_money_by_vol
    FROM instrument i
    LEFT JOIN commission_info icr ON i.instrument_id = icr.instrument_id
    WHERE i.instrument_id IN {tuple(max_open_interest['instrument_id'])}
    """
    instrument_info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    instrument_info["underlying_instr_id"] = instrument_info[
        "underlying_instr_id"
    ].apply(
        lambda x: (
            "IH" + x[2:]
            if x.startswith("HO")
            else (
                "IF" + x[2:]
                if x.startswith("IO")
                else ("IM" + x[2:] if x.startswith("MO") else x)
            )
        )
    )
    instrument_info["open_money_by_vol"] = np.where(
        instrument_info["underlying_instr_id"].str.startswith(("IH", "IF", "IM")),
        15,
        instrument_info["open_money_by_vol"],
    )
    SQL_cmd = f"""
    SELECT instrument_id as underlying_instr_id, short_margin_ratio as margin_ratio, product_id as product
    FROM instrument
    WHERE instrument_id IN {tuple(max_open_interest['underlying_instr_id'])}
    """
    margin_ratio = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    SQL_cmd = f"""
    SELECT PRODUCT_ID as product, WIND_INDUSTRYNAME1 as sector
    FROM WindIndustry_Kurt
    WHERE PRODUCT_ID IN {tuple(margin_ratio['product'])}
    """
    secinfo = pd.read_sql(sql=SQL_cmd, con=CCF.research)
    secinfo.loc[len(secinfo)] = ["IH", "金融"]
    secinfo.loc[len(secinfo)] = ["IF", "金融"]
    secinfo.loc[len(secinfo)] = ["IC", "金融"]
    secinfo.loc[len(secinfo)] = ["IM", "金融"]
    # return max_open_interest, secinfo
    SQL_cmd = f"""
    SELECT         
    instrument_id,
    prev_settlement,
    limit_up
    FROM daily_data
    WHERE trading_date = '{date[:4]}-{date[4:6]}-{date[6:]}'
    """
    limitupdf = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    limitupdf["updownlimit"] = (
        limitupdf["limit_up"] / limitupdf["prev_settlement"] - 1
    ).round(2)
    limitupdf = limitupdf[["instrument_id", "updownlimit"]]
    limitupdf = limitupdf.rename(columns={"instrument_id": "underlying_instr_id"})

    merged_df = pd.merge(
        max_open_interest, instrument_info, on="underlying_instr_id", how="left"
    )
    merged_df = pd.merge(merged_df, margin_ratio, on="underlying_instr_id", how="left")
    merged_df = pd.merge(merged_df, limitupdf, on="underlying_instr_id", how="left")
    merged_df = pd.merge(merged_df, secinfo, on="product", how="left")
    merged_df = merged_df.drop(columns=["instrument_id"]).dropna()
    merged_dict = merged_df.set_index("underlying_instr_id").T.to_dict("dict")
    for key in merged_dict:
        if merged_dict[key]["exchange_id"] == "CFFEX":
            merged_dict[key]["exchange_id"] = "CFE"
        elif merged_dict[key]["exchange_id"] == "SHFE":
            merged_dict[key]["exchange_id"] = "SHF"
        elif merged_dict[key]["exchange_id"] == "CZCE":
            merged_dict[key]["exchange_id"] = "CZC"
        elif merged_dict[key]["exchange_id"] == "GFEX":
            merged_dict[key]["exchange_id"] = "GFE"
    return merged_dict

def parse_option_contract(contract):
    # 查找第一个数字的位置
    first_digit_index = next(
        (i for i, char in enumerate(contract) if char.isdigit()), None
    )
    if first_digit_index is None:
        raise ValueError(f"无法解析合约: {contract}，未找到数字。")
    product = contract[:first_digit_index]

    if "-" in contract:
        # 处理形如 a2505-C-3850 的合约
        parts = contract.split("-")
        underlying_id = parts[0]
        type_ = parts[1]
        k = float(parts[2])
    else:
        # 处理形如 zn2505P26000 的合约
        type_index = None
        for i in range(1, len(contract) - 1):
            if (
                contract[i].isalpha()
                and contract[i - 1].isdigit()
                and contract[i + 1].isdigit()
            ):
                type_index = i
                break

        if type_index is None:
            raise ValueError(f"无法解析合约: {contract}，请检查合约格式。")

        underlying_id = contract[:type_index]
        type_ = contract[type_index]
        k = float(contract[type_index + 1 :])

    # 修改 underlying_id
    if underlying_id.startswith("HO"):
        underlying_id = "IH" + underlying_id[2:]
        product = "IH"
    elif underlying_id.startswith("IO"):
        underlying_id = "IF" + underlying_id[2:]
        product = "IF"
    elif underlying_id.startswith("MO"):
        underlying_id = "IM" + underlying_id[2:]
        product = "IM"

    return product, underlying_id, type_.lower(), k

In [5]:
#### 全量信息，输入为日期，输出为全量信息
def findOptionPrice(date):
    SQL_cmd = f"""SELECT instrument_id, trading_date, close, low, volume, open_interest, total_turnover, underlying_instr_id
    FROM daily_data_option
    WHERE trading_date = '{date}'
    """
    df = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    df["underlying_instr_id"] = df["underlying_instr_id"].apply(
        lambda x: (
            "IH" + x[2:]
            if x.startswith("HO")
            else (
                "IF" + x[2:]
                if x.startswith("IO")
                else ("IM" + x[2:] if x.startswith("MO") else x)
            )
        )
    )

    datebar = date[:4] + "-" + date[4:6] + "-" + date[6:]
    SQL_cmd = f"""SELECT instrument_id, trading_date, close
    FROM daily_data
    WHERE trading_date = '{datebar}'
    """
    unddf = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    unddf["trading_date"] = unddf["trading_date"].apply(
        lambda x: x[:4] + x[5:7] + x[8:10]
    )

    df = pd.merge(
        df,
        unddf,
        left_on=["underlying_instr_id", "trading_date"],
        right_on=["instrument_id", "trading_date"],
        how="left",
    )
    df = df.rename(columns={"close_y": "underlying_close", "close_x": "option_close", "low": "option_low"})
    df = df.drop(columns=["instrument_id_y"])
    df = df.rename(columns={"instrument_id_x": "instrument_id"})

    info_dict = findUnderlyingOptionInfo(date)

    invalid_rows = []
    error_set = []
    # 遍历每一行
    for index, row in df.iterrows():
        try:
            prod, underlying_id, type_, k = parse_option_contract(row["instrument_id"])
            if (type_ == "c" and row["underlying_close"] > k) or (
                type_ == "p" and row["underlying_close"] < k
            ):
                invalid_rows.append(index)
                continue
            expiredate = info_dict[underlying_id]["expiredate"]
            exchange = info_dict[underlying_id]["exchange_id"]
            commission = info_dict[underlying_id]["open_money_by_vol"]
            margin_ratio = info_dict[underlying_id]["margin_ratio"]
            multi = info_dict[underlying_id]["volume_multiple"]
            tick = info_dict[underlying_id]["price_tick"]
            sector = info_dict[underlying_id]["sector"]
            updownlimit = info_dict[underlying_id]["updownlimit"]

            ttm = len(
                CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= expiredate)]
            )
            df.at[index, "product"] = prod
            df.at[index, "sector"] = sector
            df.at[index, "opt_typ"] = type_
            df.at[index, "strike"] = k
            df.at[index, "tdtm"] = ttm
            df.at[index, "cdtm"] = (
                datetime.strptime(expiredate, "%Y%m%d")
                - datetime.strptime(date, "%Y%m%d")
            ).days
            df.at[index, "commission"] = commission
            df.at[index, "multiplier"] = multi
            df.at[index, "tick"] = tick
            df.at[index, "expiredate"] = expiredate
            df.at[index, "updownlimit"] = updownlimit
            df.at[index, "windcode"] = row["instrument_id"].upper() + "." + exchange
            iv = CCF.IV(
                row["option_close"], row["underlying_close"], k, ttm / 243, 0, type_
            )
            df.at[index, "iv"] = iv
            df.at[index, "delta"] = CCF.BSMgreeks.delta(
                type_, row["underlying_close"], k, ttm / 243, 0, iv, 0
            )
            otmvalue = (
                row["underlying_close"] * multi - k * multi
                if type_ == "p"
                else k * multi - row["underlying_close"] * multi
            )
            otmvalue = max(otmvalue, 0)
            orimargin = row["underlying_close"] * multi * margin_ratio
            df.at[index, "margin"] = row["option_close"] * multi + max(
                orimargin - 0.5 * otmvalue, 0.5 * orimargin
            )
        except Exception as e:
            # print(f"Error processing {row['instrument_id']}: {e}")
            invalid_rows.append(index)
            error_set.append((index, row["instrument_id"], ValueError))
    # print(df.columns)
    df = df.drop(invalid_rows).dropna().reset_index(drop=True)
    df[["volume", "open_interest", "total_turnover", "tdtm", "cdtm", "multiplier"]] = df[
        ["volume", "open_interest", "total_turnover", "tdtm", "cdtm", "multiplier"]
    ].astype(int)

    return df

In [15]:
df = findOptionPrice('20250707')
df = df.assign(margin=df['margin'])
thisdf = df.loc[df.groupby('product')['margin'].idxmin()].reset_index(drop=True)
thisdf = thisdf[['product', 'underlying_close', 'sector', 'multiplier', 'tick', 'updownlimit', 'margin', 'commission']]
sql_cmd = f"""
SELECT *
FROM Gap_Ratio 
"""
rvdf = pd.read_sql(sql_cmd, con=CCF.research)
rvdf["dailyret"] = rvdf["close_main"] / rvdf["lastday_close"] - 1
rvdf["dailyrv"] = np.square(rvdf["dailyret"])
rvdf["minrv"] = rvdf["intraday_var"] + rvdf["gap_var"]
rvdf["gapRatio"] = rvdf["gap_var"] / rvdf["minrv"]

def calpara(df):
    df = df.sort_values("trading_date")
    miniAnnualRV = np.sqrt(df.tail(30)["minrv"].mean() * 243)
    gapRatio = df.tail(30)["gap_var"].sum() / df.tail(30)["minrv"].sum()
    kurtosisShort = df.tail(240)["dailyret"].kurt()
    kurtosisLong = df["dailyret"].kurt()
    return pd.Series(
        [miniAnnualRV, gapRatio],
        index=["1minAnnualRV", "gapRatio"],
    )

rvsum = rvdf.groupby("product_id").apply(calpara).reset_index()
merged_df = pd.merge(thisdf, rvsum, left_on="product", right_on="product_id")
merged_df = merged_df.drop(columns=["product_id"])
merged_df['dollar_exposure'] = merged_df['underlying_close'] * merged_df['multiplier']
merged_df['rvexposure'] = merged_df['dollar_exposure'] * merged_df['1minAnnualRV']
merged_df['gapexposure'] = merged_df['rvexposure'] * merged_df['gapRatio']
merged_df['rvexposure_rank'] = merged_df['rvexposure'].rank(ascending=False, method='min')
merged_df['gapexposure_rank'] = merged_df['gapexposure'].rank(ascending=False, method='min')
merged_df['sellTickCredit'] = merged_df['tick'] * merged_df['multiplier'] - merged_df['commission']
### make ratio = num_long/num_short, and long as 2 ticks, short as 1 tick, exp5days = exp5t * (1-2*ratio)
threshold = 0.05
merged_df['sellTickRet'] = merged_df['sellTickCredit'] / merged_df['margin']
merged_df['exp5t'] = merged_df['sellTickRet'] / threshold
merged_df['exp5days'] = np.round(merged_df['exp5t'] * 365, 0)




Error processing HO2603-C-3100: 'IH2603'
Error processing HO2603-P-2600: 'IH2603'
Error processing HO2606-C-2900: 'IH2606'
Error processing IO2603-C-3500: 'IF2603'
Error processing IO2603-C-3900: 'IF2603'
Error processing IO2603-C-4400: 'IF2603'
Error processing IO2603-P-3200: 'IF2603'
Error processing IO2603-P-3300: 'IF2603'
Error processing IO2603-P-3500: 'IF2603'
Error processing IO2606-C-3600: 'IF2606'
Error processing IO2606-C-4300: 'IF2606'
Error processing IO2606-C-4400: 'IF2606'
Error processing IO2606-P-3400: 'IF2606'
Error processing lc2508-C-64000: cannot convert float NaN to integer
Error processing lc2508-C-65000: cannot convert float NaN to integer
Error processing lc2508-C-66000: cannot convert float NaN to integer
Error processing lc2508-C-67000: cannot convert float NaN to integer
Error processing lc2508-C-68000: cannot convert float NaN to integer
Error processing lc2508-C-70000: cannot convert float NaN to integer
Error processing lc2508-P-60000: cannot convert float

C:\Users\zhs\AppData\Local\Temp\ipykernel_25212\1630972428.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rvsum = rvdf.groupby("product_id").apply(calpara).reset_index()


In [15]:
#### 全量信息，输入为日期，输出为全量信息
def findOptionPrice2(date):
    SQL_cmd = f"""SELECT instrument_id, trading_date, close, low, volume, open_interest, total_turnover, underlying_instr_id
    FROM daily_data_option
    WHERE trading_date = '{date}'
    """
    df = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    # df["underlying_instr_id"] = df["underlying_instr_id"].apply(
    #     lambda x: (
    #         "IH" + x[2:]
    #         if x.startswith("HO")
    #         else (
    #             "IF" + x[2:]
    #             if x.startswith("IO")
    #             else ("IM" + x[2:] if x.startswith("MO") else x)
    #         )
    #     )
    # )
    return df

    datebar = date[:4] + "-" + date[4:6] + "-" + date[6:]
    SQL_cmd = f"""SELECT instrument_id, trading_date, close
    FROM daily_data
    WHERE trading_date = '{datebar}'
    """
    unddf = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    unddf["trading_date"] = unddf["trading_date"].apply(
        lambda x: x[:4] + x[5:7] + x[8:10]
    )

    df = pd.merge(
        df,
        unddf,
        left_on=["underlying_instr_id", "trading_date"],
        right_on=["instrument_id", "trading_date"],
        how="left",
    )
    df = df.rename(columns={"close_y": "underlying_close", "close_x": "option_close", "low": "option_low"})
    df = df.drop(columns=["instrument_id_y"])
    df = df.rename(columns={"instrument_id_x": "instrument_id"})

    info_dict = findUnderlyingOptionInfo(date)

    invalid_rows = []
    error_set = []
    # 遍历每一行
    for index, row in df.iterrows():
        try:
            prod, underlying_id, type_, k = parse_option_contract(row["instrument_id"])
            if (type_ == "c" and row["underlying_close"] > k) or (
                type_ == "p" and row["underlying_close"] < k
            ):
                invalid_rows.append(index)
                continue
            expiredate = info_dict[underlying_id]["expiredate"]
            exchange = info_dict[underlying_id]["exchange_id"]
            commission = info_dict[underlying_id]["open_money_by_vol"]
            margin_ratio = info_dict[underlying_id]["margin_ratio"]
            multi = info_dict[underlying_id]["volume_multiple"]
            tick = info_dict[underlying_id]["price_tick"]
            sector = info_dict[underlying_id]["sector"]
            updownlimit = info_dict[underlying_id]["updownlimit"]

            ttm = len(
                CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= expiredate)]
            )
            df.at[index, "product"] = prod
            df.at[index, "sector"] = sector
            df.at[index, "opt_typ"] = type_
            df.at[index, "strike"] = k
            df.at[index, "tdtm"] = ttm
            df.at[index, "cdtm"] = (
                datetime.strptime(expiredate, "%Y%m%d")
                - datetime.strptime(date, "%Y%m%d")
            ).days
            df.at[index, "commission"] = commission
            df.at[index, "multiplier"] = multi
            df.at[index, "tick"] = tick
            df.at[index, "expiredate"] = expiredate
            df.at[index, "updownlimit"] = updownlimit
            df.at[index, "windcode"] = row["instrument_id"].upper() + "." + exchange
            iv = CCF.IV(
                row["option_close"], row["underlying_close"], k, ttm / 243, 0, type_
            )
            df.at[index, "iv"] = iv
            df.at[index, "delta"] = CCF.BSMgreeks.delta(
                type_, row["underlying_close"], k, ttm / 243, 0, iv, 0
            )
            otmvalue = (
                row["underlying_close"] * multi - k * multi
                if type_ == "p"
                else k * multi - row["underlying_close"] * multi
            )
            otmvalue = max(otmvalue, 0)
            orimargin = row["underlying_close"] * multi * margin_ratio
            df.at[index, "margin"] = row["option_close"] * multi + max(
                orimargin - 0.5 * otmvalue, 0.5 * orimargin
            )
        except Exception as e:
            # print(f"Error processing {row['instrument_id']}: {e}")
            invalid_rows.append(index)
            error_set.append((index, row["instrument_id"], ValueError))
    # print(df.columns)
    df = df.drop(invalid_rows).dropna().reset_index(drop=True)
    df[["volume", "open_interest", "total_turnover", "tdtm", "cdtm", "multiplier"]] = df[
        ["volume", "open_interest", "total_turnover", "tdtm", "cdtm", "multiplier"]
    ].astype(int)

    return df

def test2(date):
    df = findOptionPrice2(date)
    # Vectorized approach for better performance
    def parse_and_assign(row):
        try:
            prod, _, type_, k = parse_option_contract(row["instrument_id"])
            return pd.Series({"product": prod, "opt_typ": type_, "strike": k})
        except Exception:
            return pd.Series({"product": None, "opt_typ": None, "strike": None})

    df[["product", "opt_typ", "strike"]] = df.apply(parse_and_assign, axis=1)
    df = df.dropna(subset=["product", "opt_typ", "strike"]).reset_index(drop=True)
    # For each product, select rows with underlying_instr_id that has max sum volume
    max_vol_underlying = (
        df.groupby("product")[["underlying_instr_id", "volume"]]
        .apply(lambda x: x.groupby("underlying_instr_id")["volume"].sum().idxmax())
        .reset_index()
        .rename(columns={0: "max_vol_underlying_instr_id"})
    )
    df = df.merge(max_vol_underlying[["product", "max_vol_underlying_instr_id"]], left_on="product", right_on="product")
    df = df[df["underlying_instr_id"] == df["max_vol_underlying_instr_id"]].reset_index(drop=True)
    df = df.drop(columns=["max_vol_underlying_instr_id"])
    # Prepare a tuple of instrument_ids for the SQL IN clause, ensuring correct formatting
    instrument_ids = tuple(df.groupby('underlying_instr_id').head(1)['instrument_id'])
    # If only one instrument_id, tuple will look lik
    #e ('id',) which is fine for SQL
    # Remove trailing comma in SELECT to avoid SQL syntax error
    SQL_cmd = f"""
    SELECT         
        i.underlying_instr_id,
        i.expiredate,
        i.volume_multiple,
        i.price_tick
    FROM instrument i
    WHERE i.instrument_id IN {instrument_ids}
    """
    instrument_info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    instrument_info["cdtm"] = (pd.to_datetime(instrument_info["expiredate"], format="%Y%m%d") - pd.to_datetime(date, format="%Y%m%d")).dt.days
    df = pd.merge(df, instrument_info, on="underlying_instr_id", how="left")

    # Group by 'product' and 'underlying_instr_id', then aggregate sum of 'volume', 'open_interest', and 'total_turnover'
    summary_df = (
        df.groupby(['product', 'underlying_instr_id'], as_index=False)[['volume', 'open_interest', 'total_turnover']]
        .sum()
    )
    summary_df['total_turnover_rank'] = summary_df['total_turnover'].rank(ascending=False, method='min')
    # Add a column for cdtm: take the first cdtm for each underlying_instr_id (assuming cdtm is constant per underlying)
    cdtm_map = df.groupby('underlying_instr_id')['cdtm'].first().reset_index()
    summary_df = summary_df.merge(cdtm_map, on='underlying_instr_id', how='left')

    # Add a column that counts how many rows in each underlying_instr_id have low == price_tick
    low_eq_tick_count = df[df['low'] == df['price_tick']].groupby('underlying_instr_id').size().reset_index(name='low_eq_price_tick_count')
    summary_df = summary_df.merge(low_eq_tick_count, on='underlying_instr_id', how='left')
    summary_df['low_eq_price_tick_count'] = summary_df['low_eq_price_tick_count'].fillna(0).astype(int)
    summary_df['trading_date'] = date
    return summary_df

In [16]:
date_lst = CCF.tradingDay[CCF.tradingDay<='20250718'][-100:]

In [17]:
result = pd.DataFrame()
for date in date_lst:
    result = pd.concat([result, test2(date)])

In [19]:
result.reset_index(drop=True).to_excel('./result.xlsx', index=False)

In [6]:
# Vectorized approach for better performance
def parse_and_assign(row):
    try:
        prod, _, type_, k = parse_option_contract(row["instrument_id"])
        return pd.Series({"product": prod, "opt_typ": type_, "strike": k})
    except Exception:
        return pd.Series({"product": None, "opt_typ": None, "strike": None})

df[["product", "opt_typ", "strike"]] = df.apply(parse_and_assign, axis=1)
df = df.dropna(subset=["product", "opt_typ", "strike"]).reset_index(drop=True)
# For each product, select rows with underlying_instr_id that has max sum volume
max_vol_underlying = (
    df.groupby("product")[["underlying_instr_id", "volume"]]
    .apply(lambda x: x.groupby("underlying_instr_id")["volume"].sum().idxmax())
    .reset_index()
    .rename(columns={0: "max_vol_underlying_instr_id"})
)
df = df.merge(max_vol_underlying[["product", "max_vol_underlying_instr_id"]], left_on="product", right_on="product")
df = df[df["underlying_instr_id"] == df["max_vol_underlying_instr_id"]].reset_index(drop=True)
df = df.drop(columns=["max_vol_underlying_instr_id"])
# Prepare a tuple of instrument_ids for the SQL IN clause, ensuring correct formatting
instrument_ids = tuple(df.groupby('underlying_instr_id').head(1)['instrument_id'])
# If only one instrument_id, tuple will look lik
#e ('id',) which is fine for SQL
# Remove trailing comma in SELECT to avoid SQL syntax error
SQL_cmd = f"""
SELECT         
    i.underlying_instr_id,
    i.expiredate,
    i.volume_multiple,
    i.price_tick
FROM instrument i
WHERE i.instrument_id IN {instrument_ids}
"""
instrument_info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
instrument_info["cdtm"] = (pd.to_datetime(instrument_info["expiredate"], format="%Y%m%d") - pd.to_datetime(date, format="%Y%m%d")).dt.days
df = pd.merge(df, instrument_info, on="underlying_instr_id", how="left")



In [8]:
# Group by 'product' and 'underlying_instr_id', then aggregate sum of 'volume', 'open_interest', and 'total_turnover'
summary_df = (
    df.groupby(['product', 'underlying_instr_id'], as_index=False)[['volume', 'open_interest', 'total_turnover']]
    .sum()
)
# Add a column for cdtm: take the first cdtm for each underlying_instr_id (assuming cdtm is constant per underlying)
cdtm_map = df.groupby('underlying_instr_id')['cdtm'].first().reset_index()
summary_df = summary_df.merge(cdtm_map, on='underlying_instr_id', how='left')

# Add a column that counts how many rows in each underlying_instr_id have low == price_tick
low_eq_tick_count = df[df['low'] == df['price_tick']].groupby('underlying_instr_id').size().reset_index(name='low_eq_price_tick_count')
summary_df = summary_df.merge(low_eq_tick_count, on='underlying_instr_id', how='left')
summary_df['low_eq_price_tick_count'] = summary_df['low_eq_price_tick_count'].fillna(0).astype(int)

summary_df

,product,underlying_instr_id,volume,open_interest,total_turnover,cdtm,low_eq_price_tick_count
0,AP,AP510,3695,31012.0,1841100.0,51,0
1,CF,CF509,97758,264545.0,46220200.0,37,2
2,CJ,CJ509,21720,60990.0,10351300.0,22,0
3,FG,FG508,333296,353844.0,44130500.0,4,10
4,IF,IO2507,44319,94169.0,152162220.0,11,0
5,IH,HO2507,17565,37981.0,51015360.0,11,0
6,IM,MO2507,111369,132695.0,750334540.0,11,0
7,MA,MA508,195721,209432.0,17008600.0,4,11
8,OI,OI509,23854,68785.0,16138000.0,37,0
9,PF,PF508,24591,24917.0,1396600.0,4,8


In [21]:
product_id = 'SA'
SQL_cmd = f"""SELECT TRADE_DATE, PRODUCT_ID, MAIN_ID, dailyRet_main, close_main
    FROM RY_FULL
    WHERE PRODUCT_ID = '{product_id}'
    """
info = (
    pd.read_sql(sql=SQL_cmd, con=CCF.research)
    .drop_duplicates(subset="TRADE_DATE", keep="first")
    .sort_values("TRADE_DATE")
    .dropna()
)
info["cumulative_prod"] = (info["dailyRet_main"] + 1).cumprod()
info.index = info["TRADE_DATE"]

In [25]:
info['5dayRet'] = info['cumulative_prod'] / info['cumulative_prod'].shift(5) - 1
info['20dayRet'] = info['cumulative_prod'] / info['cumulative_prod'].shift(20) - 1

In [27]:
info.dropna().sort_values('20dayRet').tail(20)

,TRADE_DATE,PRODUCT_ID,MAIN_ID,dailyRet_main,close_main,cumulative_prod,5dayRet,20dayRet
TRADE_DATE,,,,,,,,
20220208,20220208,SA,SA205,0.006402,2987.0,1.286067,0.106296,0.344890
20220210,20220210,SA,SA205,0.056814,3125.0,1.345483,0.096876,0.351644
20220207,20220207,SA,SA205,0.030556,2968.0,1.277886,0.105400,0.359597
20231215,20231215,SA,SA405,0.010531,2207.0,2.081756,-0.053602,0.366689
20231124,20231124,SA,SA401,-0.012750,2323.0,1.676976,0.100948,0.376185
20231129,20231129,SA,SA401,0.032915,2385.0,1.721734,0.018795,0.378613
20231122,20231122,SA,SA401,0.061197,2341.0,1.689970,0.138619,0.383570
20231214,20231214,SA,SA405,-0.013550,2184.0,2.060062,-0.041685,0.389319
20231130,20231130,SA,SA401,0.023899,2442.0,1.762882,0.037824,0.405063


In [24]:
### 针对SA401做20231102-20231205的一个月尾部期权研究


0.5972382048331415